# 03 · Un detector público frente a dos mamografías

**Actividad opcional · 10–15 minutos**

Ya viste las anotaciones del conjunto de datos en el cuaderno 02. Ahora veremos la salida de un modelo público sobre **los dos casos de masas**. El modelo dibuja recuadros; las zonas rojas son las máscaras anotadas por CBIS-DDSM. Son dos cosas distintas.

Usamos el modelo YOLO11n de [Digital Eye for Mammography](https://github.com/cbddobvyz/digitaleye-mammography/releases/tag/shared-models.v2). Se entrenó con otras imágenes y busca **masas**. No está preparado aquí para detectar calcificaciones. Sus etiquetas `BIRADS45` y `BIRADS12` son categorías aprendidas por el modelo, no un diagnóstico ni el resultado de patología. La cifra de confianza tampoco es una probabilidad de cáncer.

Esta demostración sirve para entender cómo se compara una predicción con una anotación. No sirve para evaluar la precisión clínica del modelo.


## 1. Preparación

Ejecuta esta celda. En Colab, sube `taller_colab.zip` si aparece la solicitud. El ZIP incluye los DICOM seleccionados y el modelo; no necesitas subir el conjunto completo.


In [ ]:
from pathlib import Path
import hashlib
import io
import json
import subprocess
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

try:
    import pydicom
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pydicom>=3,<4", "-q"])
    import pydicom

try:
    from ultralytics import YOLO
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics>=8.4,<9", "-q"])
    from ultralytics import YOLO

ubicaciones = [Path("."), Path(".."), Path("/content")]
carpeta = next(
    (ruta for ruta in ubicaciones
     if (ruta / "data/taller/casos.json").exists()
     and (ruta / "models/digitaleye_yolo11_n.pt").exists()),
    None,
)

if carpeta is None:
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError("No encuentro los datos y el modelo del taller.") from error

    print("Sube taller_colab.zip")
    archivos = files.upload()
    nombre_zip = next((nombre for nombre in archivos if nombre.endswith(".zip")), None)
    if nombre_zip is None:
        raise ValueError("Se necesita taller_colab.zip para continuar.")

    with zipfile.ZipFile(io.BytesIO(archivos[nombre_zip])) as paquete:
        rutas = [Path(nombre) for nombre in paquete.namelist()]
        if any(ruta.is_absolute() or ".." in ruta.parts for ruta in rutas):
            raise ValueError("El ZIP contiene rutas no válidas.")
        paquete.extractall("/content")
    carpeta = Path("/content")

casos = json.loads((carpeta / "data/taller/casos.json").read_text(encoding="utf-8"))
ruta_modelo = carpeta / "models/digitaleye_yolo11_n.pt"
huella_esperada = "4c6ed69c85775bfe7642a428489fc0a90f53cb3ac7b164eb7da7220ef9549a14"
huella = hashlib.sha256(ruta_modelo.read_bytes()).hexdigest()
if huella != huella_esperada:
    raise ValueError("El archivo del modelo no coincide con el verificado para este taller.")

modelo = YOLO(str(ruta_modelo))
print("Preparación lista. Modelo público de masas cargado.")


## 2. Comparar el recuadro y la referencia

El **recuadro amarillo** lo propone el modelo. La **zona roja** viene del archivo de referencia de CBIS-DDSM. Mostramos solo la propuesta con la mayor puntuación en cada imagen para que la comparación sea fácil de leer. El modelo puede haber producido otras propuestas.

La imagen se ajusta en contraste para verla en pantalla; el DICOM guardado no cambia. La predicción se calcula antes de añadir la máscara roja.


In [ ]:
def preparar_imagen(ruta):
    dicom = pydicom.dcmread(ruta)
    pixeles = dicom.pixel_array.astype(np.float32)
    visibles = pixeles[pixeles > 0]
    bajo, alto = np.percentile(visibles, [0.5, 99.7])
    imagen = np.clip((pixeles - bajo) / max(alto - bajo, 1), 0, 1)
    if dicom.PhotometricInterpretation == "MONOCHROME1":
        imagen = 1 - imagen
    return (imagen * 255).astype(np.uint8)


def comparar(caso):
    imagen = preparar_imagen(carpeta / "data/taller" / caso["imagen"])
    mascara_dicom = pydicom.dcmread(carpeta / "data/taller" / caso["mascara"])
    mascara = mascara_dicom.pixel_array > 0
    if mascara.shape != imagen.shape:
        raise ValueError("La máscara y la imagen no tienen el mismo tamaño.")

    # Tres canales idénticos permiten pasar la mamografía en grises a YOLO.
    entrada = np.repeat(imagen[:, :, None], 3, axis=2)
    resultado = modelo.predict(entrada, imgsz=1024, conf=0.05, device="cpu", verbose=False)[0]
    propuestas = resultado.boxes

    fig, ax = plt.subplots(figsize=(7, 10))
    ax.imshow(imagen, cmap="gray", vmin=0, vmax=255)
    ax.imshow(np.ma.masked_where(~mascara, mascara), cmap="Reds", alpha=0.45,
              vmin=0, vmax=1)

    if len(propuestas):
        mejor = int(propuestas.conf.argmax())
        x1, y1, x2, y2 = propuestas.xyxy[mejor].cpu().numpy()
        puntuacion = float(propuestas.conf[mejor])
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="yellow", linewidth=3))
        print(f"{caso['id']}: {len(propuestas)} propuesta(s); se muestra la de mayor puntuación ({puntuacion:.2f}).")
    else:
        print(f"{caso['id']}: no hay propuestas con el umbral elegido.")

    ax.set_title(f"{caso['id']} · amarillo: modelo / rojo: referencia")
    ax.axis("off")
    plt.show()


### Caso 1 · masa

In [ ]:
comparar(casos[0])


### Caso 2 · masa

In [ ]:
comparar(casos[1])


## ¿Qué muestran estos dos ejemplos?

En estas imágenes puedes observar si el recuadro amarillo cae cerca de la zona roja. Eso ilustra una comparación visual, **no una medida fiable de rendimiento**: dos casos son demasiado pocos para evaluar un detector.

El caso de calcificaciones del cuaderno 02 queda fuera de esta demostración porque el modelo elegido busca masas. Además, un recuadro indica una región propuesta: no reemplaza la máscara, el informe radiológico ni el diagnóstico.

**Procedencia del modelo:** [Digital Eye for Mammography](https://github.com/cbddobvyz/digitaleye-mammography), [pesos públicos `shared-models.v2`](https://github.com/cbddobvyz/digitaleye-mammography/releases/tag/shared-models.v2), licencia GPL-3.0. **Procedencia de las imágenes y máscaras:** [CBIS-DDSM en The Cancer Imaging Archive](https://www.cancerimagingarchive.net/collection/cbis-ddsm/), CC BY 3.0.
